In [1]:
import os
while not os.path.isdir("DataTypes"):
    os.chdir("..")

import json

load("DataTypes/Grafo2Terminales.sage")
load("Scripts/CantidadesEstructurales.sage")
load("Scripts/Esqueleto.sage")

In [2]:
ARCHIVO_MAXIMIZANTES = "Datos/MaximizantesT2.json"

def CargarMaximizantes(archivo=ARCHIVO_MAXIMIZANTES):
    """
        Retorna {n: {m: {"t2_max": int, "maximizantes": [Grafo2Terminales, ...]}}}
        con claves enteras, reconstruyendo cada Grafo2Terminales desde graph6.
    """
    with open(archivo) as f:
        crudo = json.load(f)
    datos = {}
    for n, por_m in crudo.items():
        for m, d in por_m.items():
            datos.setdefault(int(n), {})[int(m)] = {
                "t2_max": d["t2_max"],
                "maximizantes": [Grafo2Terminales(Graph(e["graph6"]), e["s"], e["t"])
                                 for e in d["maximizantes"]],
            }
    return datos

datos = CargarMaximizantes()

In [3]:
def EsCasiRegular(G):
    """ Grado máximo y grado mínimo difieren en 1 o menos. """
    grados = G.degree()
    return max(grados) - min(grados) <= 1

def EsMultipartitoCompleto(G):
    """
        G es multipartito completo sii la relación "no adyacentes o iguales"
        es de equivalencia, es decir, sii el complemento es unión disjunta
        de cliques (las partes). Incluye a los completos K_n (partes de tamaño 1).
    """
    return all(C.is_clique() for C in G.complement().connected_components_subgraphs())

In [4]:
def AnalizarMaximizantes(n, m):
    d = datos[n][m]
    fila = {"X": len(d["maximizantes"]), "t2": d["t2_max"],
            "casi_regular": 0, "multipartito": 0, "puentes": {}}
    for TC in d["maximizantes"]:
        esqueleto = ObtenerEsqueleto(TC).grafo
        fila["casi_regular"] += EsCasiRegular(esqueleto)
        fila["multipartito"] += EsMultipartitoCompleto(esqueleto)
        k = b(TC)
        fila["puentes"][k] = fila["puentes"].get(k, 0) + 1
    return fila

columnas = ["(n, m)", "max t2", "X", "casi-regular", "multipartito", "0 puentes", "1 puente", "otros"]
anchos = [9, 8, 3, 12, 12, 9, 8, 5]
encabezado = " | ".join(c.rjust(w) for c, w in zip(columnas, anchos))
print(encabezado)
print("-" * len(encabezado))
for n in sorted(datos):
    for m in sorted(datos[n]):
        f = AnalizarMaximizantes(n, m)
        p = f["puentes"]
        otros = sum(v for k, v in p.items() if k not in (0, 1))
        print(f"{str((n, m)):>9} | {f['t2']:>8} | {f['X']:>3} | {f['casi_regular']:>12} | "
              f"{f['multipartito']:>12} | {p.get(0, 0):>9} | {p.get(1, 0):>8} | {otros:>5}")

   (n, m) |   max t2 |   X | casi-regular | multipartito | 0 puentes | 1 puente | otros
---------------------------------------------------------------------------------------
  (7, 11) |      225 |   7 |            2 |            0 |         2 |        5 |     0
  (7, 12) |      384 |   3 |            1 |            0 |         2 |        1 |     0
  (8, 16) |     2800 |   8 |            1 |            0 |         5 |        3 |     0
  (8, 17) |     4200 |  16 |            4 |            1 |        10 |        6 |     0
  (9, 22) |    42000 |  35 |            3 |            1 |        24 |       11 |     0
  (9, 23) |    58800 |  36 |            4 |            0 |        27 |        9 |     0
 (10, 29) |   746496 |  45 |            2 |            0 |        36 |        9 |     0
 (10, 30) |  1000188 |  16 |            1 |            0 |        13 |        3 |     0
